# Absolute Lagemessung mittels LIDAR und Referenzmarken

Jonas Frei, 14.10.2026

In [ ]:
import numpy as np

## Homogene Transformation {G}->{R}

Diese Transformationsmatrix ist das gesuchte Endresultat und wäre in Echt nicht bekannt. Wir brauchen Sie jedoch als Referenzwert, sowie zum Berechnen der echten Markerposition im globalen Raum.

In [ ]:
rGR = np.array([1.5, 2, 1])
alpha = 0.5
TGR = np.array([[np.cos(alpha), -np.sin(alpha), rGR[0]],
                 [np.sin(alpha),  np.cos(alpha), rGR[1]],
                 [0,              0,             1]])

## Referenzmarkerpositionen

In [ ]:
rR1 = np.array([1, 1, 1.0])
rR2 = np.array([1, -1, 1.0])
rR3 = np.array([0.5, 0, 1.0])
rR4 = np.array([0.5, 1, 1.0])
rR5 = np.array([0, 2, 1.0])
rR6 = np.array([-3, -1, 1.0])

rG1 = TGR@rR1
rG2 = TGR@rR2
rG3 = TGR@rR3
rG4 = TGR@rR4
rG5 = TGR@rR5
rG6 = TGR@rR6

## Hinzufügen von Messrauschen

In [ ]:
def noise(maxAmplitude=0.1):
    return np.append((np.random.rand(2)*2 - 1)*maxAmplitude, 0)


rR1 = rR1 + noise()
rR2 = rR2 + noise()
rR3 = rR3 + noise()
rR4 = rR4 + noise()
rR5 = rR5 + noise()
rR6 = rR6 + noise()

## Schätzen der homogenen Transformationsmatrix

In [ ]:
def mrdivide(A, B):
    """Matrix right division, solves A * X = B for X.

    The result is exakt if A is square and invertible,
    otherwise it returns the least-squares solution.
    """
    return A @ np.linalg.pinv(B)

Aus 3 Markerpositionen

In [ ]:
TGR_est3 = mrdivide(np.column_stack([rG1, rG2, rG3]),
                     np.column_stack([rR1, rR2, rR3]))

Aus 6 Markerpositionen

In [ ]:
TGR_est6 = mrdivide(np.column_stack([rG1, rG2, rG3, rG4, rG5, rG6]),
                     np.column_stack([rR1, rR2, rR3, rR4, rR5, rR6]))

## Auswertung

In [ ]:
print("Wahre Transformationsmatrix")
print(TGR)
print("\nTransformationsmatrix geschätzt aus 3 Referenzmarkerpositionen")
print(TGR_est3)
print("\nTransformationsmatrix geschätzt aus 6 Referenzmarkerpositionen")
print(TGR_est6)

Zusätzlich zur Transformationsmatrix interessiert oft direkt der Drehwinkel $\alpha$, der sich aus den Rotationsanteilen der Matrix zurückrechnen lässt. Da die geschätzte Matrix wegen des Rauschens keine exakte Rotationsmatrix mehr ist, weichen $T_{11}$ und $T_{22}$ unabhängig voneinander leicht von $\cos\alpha$ ab, ebenso $T_{21}$ und $-T_{12}$ von $\sin\alpha$. Der Mittelwert über beide Terme nutzt die Information aus der ganzen Matrix und ist deshalb robuster gegenüber dem Rauschen als nur ein einzelnes Element:

$$\cos\alpha \approx \frac{T_{11}+T_{22}}{2}, \qquad \sin\alpha \approx \frac{T_{21}-T_{12}}{2}$$

In [ ]:
def recover_angle(T):
    cos_alpha = (T[0, 0] + T[1, 1])/2
    sin_alpha = (T[1, 0] - T[0, 1])/2
    return np.arctan2(sin_alpha, cos_alpha)


alpha_est3 = recover_angle(TGR_est3)
alpha_est6 = recover_angle(TGR_est6)

print(f"Wahrer Winkel alpha: {alpha:.4f} rad")
print(f"Geschätzter Winkel aus 3 Referenzmarkerpositionen: {alpha_est3:.4f} rad")
print(f"Geschätzter Winkel aus 6 Referenzmarkerpositionen: {alpha_est6:.4f} rad")